# Notebook 13 — Boundary Effects: Pre-15 Prime-Lane Pressure

**Repo:** `mod30-residue-lanes`  
**Layer:** `notebooks/`

Notebook 01 established the symmetric admissible-lane baseline.

Notebook 07 introduced prime-filtered asymmetry.

Notebook 11 introduced rolling window drift.

Notebook 13 focuses on the pre-15 boundary lane `13`, comparing it to its reflected partner `17` and measuring local boundary pressure around the central split:

```text
... 11 → 13 | 17 → 19 ...
```

Constraint view:

> Lane 13 marks a pre-15 boundary position where rolling prime-lane trajectories can be compared against reflected post-15 behavior.


## Goals

1. Detect repo root and create standard output directories.
2. Load MRL foundations from Notebook 00 when available.
3. Generate prime-filtered admissible-lane data.
4. Build rolling windows over prime lane counts.
5. Focus on lane `13` and reflected lane `17`.
6. Measure pre/post boundary imbalance, reflection ratio, local pressure, and boundary drift.
7. Export CSV, JSON, figures, and Markdown report.
8. Create a Colab-downloadable output zip.


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/mod30-residue-lanes"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "src" / "mod30_residue_lanes").exists() or (c / "notebooks").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
REPORTS_DIR = REPO_ROOT / "reports"

for d in [NOTEBOOKS_DIR, RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("REPORTS_DIR:", REPORTS_DIR)


## Load Notebook 00 foundation output

In [ ]:
foundation_path = RESULTS_DIR / "notebook00_mrl_foundations.json"

if foundation_path.exists():
    foundation = json.loads(foundation_path.read_text())
    MODULUS = foundation["modulus"]
    ADMISSIBLE_RESIDUES = foundation["admissible_residues"]
    print("Loaded foundation:", foundation_path)
else:
    MODULUS = 30
    ADMISSIBLE_RESIDUES = [1, 7, 11, 13, 17, 19, 23, 29]
    print("Notebook 00 foundation not found; using defaults.")

TARGET_LANE = 13
REFLECTED_LANE = 17
PRE_BOUNDARY_LANES = [1, 7, 11, 13]
POST_BOUNDARY_LANES = [17, 19, 23, 29]

LANE_LABEL = f"{TARGET_LANE:02d}"
REFLECTED_LABEL = f"{REFLECTED_LANE:02d}"

print("MODULUS:", MODULUS)
print("ADMISSIBLE_RESIDUES:", ADMISSIBLE_RESIDUES)
print("TARGET_LANE:", LANE_LABEL)
print("REFLECTED_LANE:", REFLECTED_LABEL)


## Helpers

In [ ]:
def prime_sieve(n_max: int) -> np.ndarray:
    if n_max < 2:
        return np.zeros(n_max + 1, dtype=bool)

    sieve = np.ones(n_max + 1, dtype=bool)
    sieve[:2] = False

    limit = int(np.sqrt(n_max)) + 1
    for p in range(2, limit):
        if sieve[p]:
            sieve[p*p:n_max+1:p] = False

    return sieve


def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


## Generate prime-filtered lane data

In [ ]:
N_MAX = 90000
WINDOW_SIZE = 3000
STEP_SIZE = 300

values = np.arange(1, N_MAX + 1)
is_prime_array = prime_sieve(N_MAX)

df = pd.DataFrame({
    "n": values,
    "residue": values % MODULUS,
})

df["is_admissible"] = df["residue"].isin(ADMISSIBLE_RESIDUES)
df["is_prime"] = is_prime_array[values]
df["is_admissible_prime"] = df["is_admissible"] & df["is_prime"]
df["cycle"] = (df["n"] - 1) // MODULUS

admissible_prime_df = df[df["is_admissible_prime"]].copy()
lane13_prime_df = df[df["is_admissible_prime"] & (df["residue"] == TARGET_LANE)].copy()
lane17_prime_df = df[df["is_admissible_prime"] & (df["residue"] == REFLECTED_LANE)].copy()

print("Total values:", len(df))
print("Admissible prime values:", len(admissible_prime_df))
print("Lane 13 prime values:", len(lane13_prime_df))
print("Lane 17 prime values:", len(lane17_prime_df))


## Rolling boundary vectors

Boundary metrics:

- `pre_boundary_count`: lanes `01, 07, 11, 13`
- `post_boundary_count`: lanes `17, 19, 23, 29`
- `boundary_imbalance`: pre minus post
- `reflection_gap_13_17`: lane 13 minus lane 17
- `reflection_ratio_13_17`: lane 13 divided by lane 17


In [ ]:
lane_cols = [f"lane_{r:02d}" for r in ADMISSIBLE_RESIDUES]
window_rows = []

for start in range(1, N_MAX - WINDOW_SIZE + 2, STEP_SIZE):
    stop = start + WINDOW_SIZE - 1
    window = df[(df["n"] >= start) & (df["n"] <= stop)]
    prime_window = window[window["is_admissible_prime"]]
    counts = prime_window.groupby("residue").size().reindex(ADMISSIBLE_RESIDUES, fill_value=0)

    pre_count = int(counts.loc[PRE_BOUNDARY_LANES].sum())
    post_count = int(counts.loc[POST_BOUNDARY_LANES].sum())
    lane13_count = int(counts.loc[TARGET_LANE])
    lane17_count = int(counts.loc[REFLECTED_LANE])

    row = {
        "window_start": start,
        "window_stop": stop,
        "window_center": (start + stop) / 2,
        "window_size": len(window),
        "prime_count": int(prime_window.shape[0]),
        "pre_boundary_count": pre_count,
        "post_boundary_count": post_count,
        "boundary_imbalance": pre_count - post_count,
        "boundary_abs_imbalance": abs(pre_count - post_count),
        "lane13_count": lane13_count,
        "lane17_count": lane17_count,
        "reflection_gap_13_17": lane13_count - lane17_count,
        "reflection_abs_gap_13_17": abs(lane13_count - lane17_count),
        "reflection_ratio_13_17": lane13_count / max(1, lane17_count),
        "boundary_pressure": abs(pre_count - post_count) / max(1, pre_count + post_count),
        "reflection_pressure": abs(lane13_count - lane17_count) / max(1, lane13_count + lane17_count),
    }

    for residue, count in counts.items():
        row[f"lane_{residue:02d}"] = int(count)

    row["leader_lane"] = f"{int(counts.idxmax()):02d}"
    row["leader_count"] = int(counts.max())
    window_rows.append(row)

boundary_df = pd.DataFrame(window_rows)

boundary_csv = RESULTS_DIR / "notebook13_boundary_rolling_vectors.csv"
boundary_df.to_csv(boundary_csv, index=False)

print(boundary_csv)
boundary_df.head()


## Boundary drift metrics

In [ ]:
matrix = boundary_df[lane_cols].to_numpy(dtype=float)

drift_rows = []
for i in range(1, len(matrix)):
    prev_vec = matrix[i - 1]
    cur_vec = matrix[i]

    drift_rows.append({
        "window_start": int(boundary_df.loc[i, "window_start"]),
        "euclidean_drift": float(np.linalg.norm(cur_vec - prev_vec)),
        "cosine_similarity_previous": cosine_similarity(prev_vec, cur_vec),
        "cosine_drift": float(1 - cosine_similarity(prev_vec, cur_vec)),
        "boundary_imbalance_delta": float(
            boundary_df.loc[i, "boundary_imbalance"] - boundary_df.loc[i - 1, "boundary_imbalance"]
        ),
        "reflection_gap_delta": float(
            boundary_df.loc[i, "reflection_gap_13_17"] - boundary_df.loc[i - 1, "reflection_gap_13_17"]
        ),
        "boundary_pressure": float(boundary_df.loc[i, "boundary_pressure"]),
        "reflection_pressure": float(boundary_df.loc[i, "reflection_pressure"]),
    })

drift_df = pd.DataFrame(drift_rows)

drift_csv = RESULTS_DIR / "notebook13_boundary_drift_metrics.csv"
drift_df.to_csv(drift_csv, index=False)

print(drift_csv)
drift_df.head()


## Boundary pair similarity

In [ ]:
lane13_traj = boundary_df["lane13_count"].to_numpy()
lane17_traj = boundary_df["lane17_count"].to_numpy()

pair_summary = pd.DataFrame([{
    "lane_a": "13",
    "lane_b": "17",
    "cosine_similarity": cosine_similarity(lane13_traj, lane17_traj),
    "pearson_correlation": float(np.corrcoef(lane13_traj, lane17_traj)[0, 1]),
    "mean_lane13_count": float(boundary_df["lane13_count"].mean()),
    "mean_lane17_count": float(boundary_df["lane17_count"].mean()),
    "mean_reflection_gap_13_17": float(boundary_df["reflection_gap_13_17"].mean()),
    "mean_reflection_abs_gap_13_17": float(boundary_df["reflection_abs_gap_13_17"].mean()),
    "mean_reflection_pressure": float(boundary_df["reflection_pressure"].mean()),
}])

pair_csv = RESULTS_DIR / "notebook13_boundary_pair_similarity.csv"
pair_summary.to_csv(pair_csv, index=False)

print(pair_csv)
pair_summary


## Boundary leadership summary

In [ ]:
leadership_df = (
    boundary_df
    .groupby("leader_lane")
    .agg(
        leadership_windows=("leader_lane", "size"),
        mean_leader_count=("leader_count", "mean"),
    )
    .reset_index()
    .sort_values("leader_lane")
)

leadership_csv = RESULTS_DIR / "notebook13_boundary_leadership_summary.csv"
leadership_df.to_csv(leadership_csv, index=False)

print(leadership_csv)
leadership_df


## Summary exports

In [ ]:
summary = {
    "repo": "mod30-residue-lanes",
    "notebook": "13_lane_residue_13",
    "title": "Boundary Effects: Pre-15 Prime-Lane Pressure",
    "modulus": MODULUS,
    "target_lane": TARGET_LANE,
    "target_lane_label": LANE_LABEL,
    "reflected_lane": REFLECTED_LANE,
    "reflected_lane_label": REFLECTED_LABEL,
    "admissible_residues": ADMISSIBLE_RESIDUES,
    "pre_boundary_lanes": PRE_BOUNDARY_LANES,
    "post_boundary_lanes": POST_BOUNDARY_LANES,
    "n_max": N_MAX,
    "window_size": WINDOW_SIZE,
    "step_size": STEP_SIZE,
    "rolling_windows": int(len(boundary_df)),
    "admissible_prime_values": int(len(admissible_prime_df)),
    "lane13_prime_values": int(len(lane13_prime_df)),
    "lane17_prime_values": int(len(lane17_prime_df)),
    "mean_boundary_imbalance": float(boundary_df["boundary_imbalance"].mean()),
    "mean_boundary_abs_imbalance": float(boundary_df["boundary_abs_imbalance"].mean()),
    "mean_boundary_pressure": float(boundary_df["boundary_pressure"].mean()),
    "mean_reflection_gap_13_17": float(boundary_df["reflection_gap_13_17"].mean()),
    "mean_reflection_abs_gap_13_17": float(boundary_df["reflection_abs_gap_13_17"].mean()),
    "mean_reflection_pressure": float(boundary_df["reflection_pressure"].mean()),
    "mean_euclidean_drift": float(drift_df["euclidean_drift"].mean()),
    "max_euclidean_drift": float(drift_df["euclidean_drift"].max()),
    "constraint_view": "Lane 13 marks a pre-15 boundary position where rolling prime-lane trajectories can be compared against reflected post-15 behavior.",
}

json_path = RESULTS_DIR / "notebook13_boundary_effects_summary.json"
json_path.write_text(json.dumps(summary, indent=2))

print(json_path)
print(json.dumps(summary, indent=2))


## Figures

In [ ]:
figure_names = []

def savefig(name):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    figure_names.append(name)
    print(path)

# 1. Rolling boundary heatmap
matrix = boundary_df[lane_cols].to_numpy()
plt.figure(figsize=(11, 6))
plt.imshow(matrix, aspect="auto")
plt.xticks(range(len(lane_cols)), [c.replace("lane_", "") for c in lane_cols])
plt.yticks(
    range(0, len(boundary_df), max(1, len(boundary_df)//10)),
    boundary_df["window_start"].iloc[::max(1, len(boundary_df)//10)]
)
plt.xlabel("Residue lane")
plt.ylabel("Rolling window start")
plt.colorbar(label="Prime count")
plt.title("Notebook 13 — Boundary Rolling Prime Lane Vectors")
savefig("notebook13_boundary_rolling_prime_lane_vectors.png")

# 2. Lane 13 vs 17
plt.figure(figsize=(12, 5))
plt.plot(boundary_df["window_center"], boundary_df["lane13_count"], label="13")
plt.plot(boundary_df["window_center"], boundary_df["lane17_count"], label="17")
plt.xlabel("Window center")
plt.ylabel("Prime count")
plt.title("Notebook 13 — Boundary Pair: Lane 13 vs Lane 17")
plt.legend()
savefig("notebook13_lane13_vs_lane17.png")

# 3. Reflection gap
plt.figure(figsize=(12, 5))
plt.plot(boundary_df["window_center"], boundary_df["reflection_gap_13_17"])
plt.axhline(0, linewidth=1)
plt.xlabel("Window center")
plt.ylabel("Lane 13 count minus lane 17 count")
plt.title("Notebook 13 — Reflection Gap 13 − 17")
savefig("notebook13_reflection_gap_13_17.png")

# 4. Boundary imbalance
plt.figure(figsize=(12, 5))
plt.plot(boundary_df["window_center"], boundary_df["boundary_imbalance"])
plt.axhline(0, linewidth=1)
plt.xlabel("Window center")
plt.ylabel("Pre-boundary minus post-boundary count")
plt.title("Notebook 13 — Pre/Post Boundary Imbalance")
savefig("notebook13_pre_post_boundary_imbalance.png")

# 5. Boundary pressure
plt.figure(figsize=(12, 5))
plt.plot(boundary_df["window_center"], boundary_df["boundary_pressure"], label="pre/post boundary pressure")
plt.plot(boundary_df["window_center"], boundary_df["reflection_pressure"], label="13/17 reflection pressure")
plt.xlabel("Window center")
plt.ylabel("Pressure")
plt.title("Notebook 13 — Boundary and Reflection Pressure")
plt.legend()
savefig("notebook13_boundary_reflection_pressure.png")

# 6. Boundary drift
plt.figure(figsize=(12, 5))
plt.plot(drift_df["window_start"], drift_df["euclidean_drift"])
plt.xlabel("Rolling window start")
plt.ylabel("Euclidean drift")
plt.title("Notebook 13 — Boundary Vector Drift")
savefig("notebook13_boundary_vector_drift.png")

# 7. Leadership windows
plt.figure(figsize=(8, 4))
plt.bar(leadership_df["leader_lane"], leadership_df["leadership_windows"])
plt.xlabel("Leader lane")
plt.ylabel("Leadership windows")
plt.title("Notebook 13 — Boundary Leadership Windows")
savefig("notebook13_boundary_leadership_windows.png")


## Build Markdown report

The report uses repo-relative links for CSV, JSON, and figure outputs.


In [ ]:
report_path = REPORTS_DIR / "report_13_boundary_effects.md"

output_links = "\n".join([
    '- Boundary rolling vectors CSV: <a href="../results/notebook13_boundary_rolling_vectors.csv">`results/notebook13_boundary_rolling_vectors.csv`</a>',
    '- Boundary drift metrics CSV: <a href="../results/notebook13_boundary_drift_metrics.csv">`results/notebook13_boundary_drift_metrics.csv`</a>',
    '- Boundary pair similarity CSV: <a href="../results/notebook13_boundary_pair_similarity.csv">`results/notebook13_boundary_pair_similarity.csv`</a>',
    '- Boundary leadership summary CSV: <a href="../results/notebook13_boundary_leadership_summary.csv">`results/notebook13_boundary_leadership_summary.csv`</a>',
    '- Summary JSON: <a href="../results/notebook13_boundary_effects_summary.json">`results/notebook13_boundary_effects_summary.json`</a>',
] + [
    f'- Figure: <a href="../figures/{name}">`figures/{name}`</a>'
    for name in figure_names
])

report = f"""# Report 13 — Boundary Effects: Pre-15 Prime-Lane Pressure

Notebook 13 studies lane `13` as a pre-15 boundary lane and compares it against reflected lane `17`.

Constraint view:

> Lane 13 marks a pre-15 boundary position where rolling prime-lane trajectories can be compared against reflected post-15 behavior.

## Generated outputs

{output_links}

## Summary

| Metric | Value |
|---|---:|
| Modulus | {MODULUS} |
| Target lane | {LANE_LABEL} |
| Reflected lane | {REFLECTED_LABEL} |
| Interval max | {N_MAX} |
| Window size | {WINDOW_SIZE} |
| Step size | {STEP_SIZE} |
| Rolling windows | {len(boundary_df)} |
| Admissible prime values | {len(admissible_prime_df)} |
| Lane 13 prime values | {len(lane13_prime_df)} |
| Lane 17 prime values | {len(lane17_prime_df)} |
| Mean boundary imbalance | {boundary_df["boundary_imbalance"].mean():.6f} |
| Mean boundary pressure | {boundary_df["boundary_pressure"].mean():.6f} |
| Mean reflection gap 13 − 17 | {boundary_df["reflection_gap_13_17"].mean():.6f} |
| Mean reflection pressure | {boundary_df["reflection_pressure"].mean():.6f} |

## Boundary pair similarity

{pair_summary.to_markdown(index=False)}

## Boundary leadership summary

{leadership_df.to_markdown(index=False)}

## Boundary metric summary

{boundary_df[["boundary_imbalance", "boundary_pressure", "reflection_gap_13_17", "reflection_pressure"]].describe().to_markdown()}

## Interpretation

- Notebook 13 treats lane `13` as the pre-15 boundary lane.
- Lane `17` is used as the reflected post-15 comparison lane.
- Boundary imbalance measures aggregate pre-boundary vs post-boundary prime occupancy.
- Reflection pressure measures how sharply lane `13` diverges from lane `17`.
- Boundary drift measures how the full eight-lane vector changes while this central split is tracked.

## Next step

Notebook 17 can study reflected-lane behavior directly: lane `17` as the post-15 partner of lane `13`.
"""

report_path.write_text(report)
print(report_path)


## Report preview

In [ ]:
print(report_path.read_text()[:5000])


## Render generated figures in notebook

In [ ]:
from IPython.display import Image, display, Markdown

for name in figure_names:
    display(Markdown(f"### `{name}`"))
    display(Image(filename=str(FIGURES_DIR / name)))


## Create output zip

In [ ]:
EXPORT_NAME = "notebook13_boundary_effects_outputs.zip"
export_path = REPO_ROOT / EXPORT_NAME

with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.glob("notebook13_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    for p in FIGURES_DIR.glob("notebook13_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    for p in REPORTS_DIR.glob("report_13_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))

print(export_path)


## Optional Colab download

Uncomment and run this cell in Colab to download generated outputs.


In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook13_boundary_effects_outputs.zip"
# export_path = REPO_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook13_*"):
#             zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
#         for p in folder.glob("report_13_*"):
#             zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))
